# 검색 결과 평가 및 키워드 후보 추출 실험

이 노트북은 Leren의 데이터 선별 및 RAG 검색 품질을 점검하기 위한 **검증용 실험 노트북**입니다.
운영 API를 실행하거나 외부 LLM·데이터베이스에 데이터를 저장하지 않습니다.

## 이 노트북에서 확인하는 내용

1. 검색 결과에 관련 문서가 포함되는지 `Recall@K`와 `MRR`의 계산 방식을 확인합니다.
2. 한국은행 경제용어 789건에 인플레이션 키워드 필터를 적용하여 1차 후보를 추출합니다.
3. 사람 검수 라벨이 없는 상태에서 Precision·Recall·F1을 임의로 계산하지 않는지 확인합니다.
4. 이후 사람이 작성한 Gold Label을 이용해 키워드 필터의 누락과 오탐을 평가할 수 있도록 준비합니다.

> 핵심 원칙: 키워드 일치 여부는 최종 적합성 판정이 아니라 LLM 또는 사람 검수가 이어질 후보를 넓게 모으기 위한 1차 필터입니다.

## 1. 검색 평가 지표의 간단한 예시

검색 결과는 순서가 있는 문서 ID 목록으로 표현하고, 정답으로 간주되는 관련 문서 ID 집합과 비교합니다.

- `Recall@K`: 상위 K개 검색 결과 안에 관련 문서가 하나 이상 포함되었는지 확인합니다. 튜터 검색에서 중요한 **누락 방지**를 보는 지표입니다.
- `MRR`: 첫 번째 관련 문서가 몇 번째 순위에 나왔는지 평가합니다. 관련 문서가 앞에 나올수록 점수가 높습니다.
- 이 예시에서는 `DOC-2`가 2위에 있으므로 `Recall@5`는 관련 문서를 찾았는지, `MRR`은 첫 관련 문서의 순위가 어떤지 확인합니다.

실제 서비스 평가에서는 사람이 확인한 정답 문서 집합(Gold Label)이 필요합니다. 아래 코드는 지표 함수가 의도대로 동작하는지 확인하는 최소 예시입니다.

In [ ]:
# 검색 결과 평가 함수를 불러옵니다.
from app.evaluation.rag_metrics import evaluate_retrieval

# 각 원소는 (검색 결과 ID 목록, 사람이 정한 관련 문서 ID 집합)입니다.
cases = [(["DOC-1", "DOC-2"], {"DOC-2"})]

# 상위 5개 결과를 기준으로 Recall@5와 MRR을 계산합니다.
retrieval_metrics = evaluate_retrieval(cases, k=5)
retrieval_metrics

### 지표 해석 시 주의할 점

위 코드는 평가 함수의 동작 예시일 뿐, 실제 Leren 검색 품질의 성능 수치가 아닙니다. 실제 성능을 보고하려면 질문별로 관련 문서를 사람이 라벨링한 뒤 동일한 형식의 평가 데이터를 만들어야 합니다.

특히 관련 문서 라벨이 없는 상태에서는 Recall·Precision·F1을 계산하지 않고 `NOT_EVALUATED`로 남기는 것이 올바른 절차입니다.

## 2. 인플레이션 키워드 기반 1차 후보 추출

### 실험 목적

한국은행 RAG 데이터 789건에 기존 `keyword-v1` 후보 생성기를 적용하여, 인플레이션 챕터와 관련된 자료가 어느 정도 규모로 추출되는지 확인합니다.

이 단계의 결과는 ‘관련 문서’가 확정되었다는 의미가 아닙니다. 키워드가 포함되었지만 실제 내용은 무관할 수 있고, 키워드가 없어도 의미상 중요한 문서가 있을 수 있습니다. 따라서 후보 추출 후 LLM 정밀 분류 또는 사람 검수를 진행합니다.

### 실행 범위와 안전장치

- 입력: 한국은행 용어 데이터 789건
- 처리: 인플레이션 관련 키워드와 표현을 이용한 후보 추출
- 미실행: 외부 LLM 호출, PostgreSQL/pgvector 쓰기, 원본 데이터 수정
- 평가 상태: Gold Label이 없으므로 Precision·Recall·F1은 `NOT_EVALUATED`로 유지

In [ ]:
# 후보 생성 로직을 호출합니다. 이 함수는 로컬 파일을 읽어 결과를 반환하며,
# 이 노트북 실행만으로 LLM 호출이나 DB 저장을 수행하지 않습니다.
from scripts.evaluate_inflation_keyword import run

# 결과에는 전체 입력 건수, 후보 건수, 자동 분류 상태, 실행 안전장치 정보가 포함됩니다.
result = run()
result

In [ ]:
# 재현성을 위해 현재 실험의 입력 규모와 외부 부작용 여부를 검증합니다.
assert result["source_record_count"] == 789
# 현재 저장소의 BOK 2026 JSONL 기준 후보 수는 54건입니다.
assert result["candidate_count"] == 54
assert result["llm_calls"] == 0
assert result["db_writes"] == 0

# 발표·보고서에서 우선 확인할 요약 정보입니다.
{
    # 실험이 평가 가능한 상태인지 표시합니다.
    "status": result["status"],
    # 키워드 필터를 통과한 후보 문서 수입니다. 최종 승인 문서 수가 아닙니다.
    "candidate_count": result["candidate_count"],
    # 키워드 규칙이 부여한 임시 분류 결과의 개수입니다. Gold Label이 아닙니다.
    "prediction_label_counts": result["prediction_label_counts"],
    # 사람이 확인한 정답 라벨 수입니다. 현재는 라벨이 없으므로 0이어야 합니다.
    "labeled_count": result["labeled_count"],
    # Gold Label이 없으면 Precision/Recall/F1은 NOT_EVALUATED로 유지됩니다.
    "metrics": result["metrics"],
}

## 3. 다음 평가 단계

현재 노트북의 후보 수 54건은 키워드 필터의 처리 결과이지, 모델의 정확도나 최종 승인 건수가 아닙니다. 데이터 파일이나 키워드 규칙이 바뀌면 이 수치도 달라질 수 있습니다. 다음 단계에서는 사람이 원문을 확인하여 Gold Label을 작성해야 합니다.

### 검수 절차

1. `reports/step3/gold_label_template.csv`에서 후보·경계·무작위 표본의 원문을 확인합니다.
2. `gold_label` 열에 `approved`, `needs_review`, `irrelevant` 중 하나를 직접 입력합니다.
3. 자동 예측값을 정답 라벨로 복사하지 않습니다. 그래야 키워드 필터의 성능을 독립적으로 평가할 수 있습니다.
4. `scripts/evaluate_step3_gold.py`를 실행하여 Precision·Recall·F1과 누락 사례를 확인합니다.

### 표본 범위

`reports/step3/gold_label_template.csv`는 후보·경계·무작위 표본 205건을 포함합니다. 이 표본으로는 후보 필터의 초기 품질을 점검할 수 있지만, 전체 모집단 789건의 정확한 Recall을 산출하려면 `reports/step3/full_corpus_gold_template.csv`에 전체 정답 라벨을 작성해야 합니다.

이후에는 Gold Label을 기준으로 키워드 범위와 챕터별 임계값을 조정하고, 최종 후보에만 비용이 큰 LLM 정밀 분류를 적용합니다.